# 04 — Optuna Tuning

ini ganti Grid Search dengan **Tree-structured Parzen Estimator (Optuna)** untuk mengeksplorasi ruang parameter yang lebih luas dengan jumlah trial lebih sedikit.

- `learning_rate` (log-uniform 1e-4 .. 1e-1)
- `optimizer ∈ {adam, rmsprop}`
- `lstm_units ∈ {16, 32, 64, 128}`
- `dropout_rate ∈ [0, 0.4]` step 0.05

Output: best params per stasiun + perbandingan terhadap Grid Search dari notebook 03.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

from src import config as C
from src.data_loader import load_all_stations
from src.evaluation import compute_metrics
from src.preprocessing import build_pipeline, inverse_target
from src.tuning_optuna import run_optuna, fit_best

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FEATURES = C.WEATHER_COLS + [C.AOD_COL, C.TARGET]
STATIONS_TO_RUN = C.HEALTHY_STATIONS
N_TRIALS = 30
EPOCHS = 100
PATIENCE = 15

dfs = load_all_stations(reindex=True)

In [3]:
rows = []
for s in STATIONS_TO_RUN:
    print(f'\n=== {s} ===')
    data = build_pipeline(dfs[s], FEATURES, smooth_aod=False)

    study = run_optuna(data, n_trials=N_TRIALS, epochs=EPOCHS, patience=PATIENCE)
    model, history = fit_best(study, data, epochs=EPOCHS, patience=PATIENCE)

    n_features = data['X_train'].shape[2]
    y_pred = model.predict(data['X_test'], verbose=0).flatten()
    y_pred = inverse_target(y_pred, data['scaler'], data['target_idx'], n_features)
    y_true = inverse_target(data['y_test'], data['scaler'], data['target_idx'], n_features)
    m = compute_metrics(y_true, y_pred)

    rows.append({'station': s, **study.best_params, **m})
    model.save(C.MODEL_DIR / f'04_optuna_{s}.keras')
    print(f'   best R²={m["R2"]:.4f} | {study.best_params}')

df_optuna = pd.DataFrame(rows)
df_optuna.to_csv(C.METRICS_DIR / '04_optuna_summary.csv', index=False)
df_optuna

[I 2026-04-27 13:10:42,095] A new study created in memory with name: no-name-5fa33f73-6a21-4060-9303-081bbdc61411



=== us_embassy_1 ===


[I 2026-04-27 13:16:44,121] Trial 0 finished with value: 0.442237515910923 and parameters: {'learning_rate': 0.0005611516415334506, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.35000000000000003}. Best is trial 0 with value: 0.442237515910923.
[I 2026-04-27 13:20:25,419] Trial 1 finished with value: 0.4968690978894894 and parameters: {'learning_rate': 0.0015930522616241021, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.05}. Best is trial 1 with value: 0.4968690978894894.


[I 2026-04-27 13:23:42,524] Trial 2 finished with value: 0.4715158120042381 and parameters: {'learning_rate': 0.0004059611610484307, 'optimizer': 'adam', 'lstm_units': 32, 'dropout_rate': 0.15000000000000002}. Best is trial 1 with value: 0.4968690978894894.
[I 2026-04-27 13:26:38,345] Trial 3 finished with value: 0.3283549292467367 and parameters: {'learning_rate': 0.000816845589476017, 'optimizer': 'adam', 'lstm_units': 128, 'dropout_rate': 0.05}. Best is trial 1 with value: 0.4968690978894894.
[I 2026-04-27 13:27:47,685] Trial 4 finished with value: 0.011337041027407402 and parameters: {'learning_rate': 0.00013492834268013249, 'optimizer': 'rmsprop', 'lstm_units': 16, 'dropout_rate': 0.15000000000000002}. Best is trial 1 with value: 0.4968690978894894.
[I 2026-04-27 13:28:57,404] Trial 5 finished with value: 0.05110082628556023 and parameters: {'learning_rate': 0.00017541893487450815, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.2}. Best is trial 1 with value: 0.496869097

   best R²=-3743226.9185 | {'learning_rate': 0.0003074326026550764, 'optimizer': 'adam', 'lstm_units': 128, 'dropout_rate': 0.0}

=== us_embassy_2 ===


...

[I 2026-04-27 22:50:04,400] Trial 0 finished with value: 0.18597198712211027 and parameters: {'learning_rate': 0.0005611516415334506, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.35000000000000003}. Best is trial 0 with value: 0.18597198712211027.
[I 2026-04-27 22:51:38,235] Trial 1 finished with value: 0.4440480961682547 and parameters: {'learning_rate': 0.0015930522616241021, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.05}. Best is trial 1 with value: 0.4440480961682547.
[I 2026-04-27 22:53:24,779] Trial 2 finished with value: 0.31866622334980366 and parameters: {'learning_rate': 0.0004059611610484307, 'optimizer': 'adam', 'lstm_units': 32, 'dropout_rate': 0.15000000000000002}. Best is trial 1 with value: 0.4440480961682547.
[I 2026-04-27 22:55:26,198] Trial 3 finished with value: 0.4207243632023583 and parameters: {'learning_rate': 0.000816845589476017, 'optimizer': 'adam', 'lstm_units': 128, 'dropout_rate': 0.05}. Best is trial 1 with value: 0.4440480961682

   best R²=-0.0536 | {'learning_rate': 0.0015930522616241021, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.05}

=== kebun_jeruk ===


[I 2026-04-27 23:40:01,332] Trial 0 finished with value: 0.039525022923620035 and parameters: {'learning_rate': 0.0005611516415334506, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.35000000000000003}. Best is trial 0 with value: 0.039525022923620035.
[I 2026-04-27 23:41:49,346] Trial 1 finished with value: 0.5540952740380163 and parameters: {'learning_rate': 0.0015930522616241021, 'optimizer': 'adam', 'lstm_units': 16, 'dropout_rate': 0.05}. Best is trial 1 with value: 0.5540952740380163.
[I 2026-04-27 23:43:47,446] Trial 2 finished with value: 0.43060066132097885 and parameters: {'learning_rate': 0.0004059611610484307, 'optimizer': 'adam', 'lstm_units': 32, 'dropout_rate': 0.15000000000000002}. Best is trial 1 with value: 0.5540952740380163.
[I 2026-04-27 23:45:33,094] Trial 3 finished with value: 0.5470499259887422 and parameters: {'learning_rate': 0.000816845589476017, 'optimizer': 'adam', 'lstm_units': 128, 'dropout_rate': 0.05}. Best is trial 1 with value: 0.55409527403

   best R²=0.5202 | {'learning_rate': 0.0006990551259891295, 'optimizer': 'adam', 'lstm_units': 64, 'dropout_rate': 0.0}


,station,learning_rate,optimizer,lstm_units,dropout_rate,R2,MSE,RMSE,MAE
0,us_embassy_1,0.000307,adam,128,0.00,-3.743227e+06,1.664473e+09,40797.947769,19618.684208
1,us_embassy_2,0.000447,rmsprop,128,0.10,-1.621711e-03,6.437204e+02,25.371646,21.658303
2,bundaran_hi,0.000817,adam,128,0.05,6.413085e-01,3.234068e+02,17.983516,13.357601
3,kelapa_gading,0.000421,adam,128,0.05,6.460235e-01,1.875588e+02,13.695210,10.674044
4,jagakarsa,0.001113,adam,128,0.00,5.259171e-01,1.608693e+02,12.683427,8.429898
5,lubang_buaya,0.001593,adam,16,0.05,-5.356521e-02,6.385809e+02,25.270158,20.310780
6,kebun_jeruk,0.000699,adam,64,0.00,5.202040e-01,4.567227e+02,21.371072,17.561974


## Perbandingan Grid Search vs Optuna

In [4]:
df_grid = pd.read_csv(C.METRICS_DIR / '03_grid_best_summary.csv')[['station', 'R2']].rename(columns={'R2': 'R2_grid'})
df_opt  = df_optuna[['station', 'R2']].rename(columns={'R2': 'R2_optuna'})
comp = df_grid.merge(df_opt, on='station')
comp['delta'] = comp['R2_optuna'] - comp['R2_grid']
comp.to_csv(C.METRICS_DIR / '04_grid_vs_optuna.csv', index=False)
comp

,station,R2_grid,R2_optuna,delta
0,kelapa_gading,0.634426,6.460235e-01,1.159701e-02
1,bundaran_hi,0.596756,6.413085e-01,4.455226e-02
2,kebun_jeruk,0.591850,5.202040e-01,-7.164647e-02
3,jagakarsa,0.488738,5.259171e-01,3.717865e-02
4,us_embassy_2,0.415823,-1.621711e-03,-4.174443e-01
5,lubang_buaya,0.017091,-5.356521e-02,-7.065654e-02
6,us_embassy_1,-0.217306,-3.743227e+06,-3.743227e+06
